In [1]:
# ===============================
# 0. 基础依赖
# ===============================
import os
import copy
import pandas as pd
import numpy as np
import networkx as nx

In [3]:
# ===============================
# 1. 读取 host / virus 指标
# ===============================
def load_host_virus_indices(
    host_file,
    virus_file,
    host_col='host',
    virus_col='virus',
    hii_col='HII',
    psi_col='PSI'
):
    host_df = pd.read_excel(host_file)
    virus_df = pd.read_excel(virus_file)

    host_hii = dict(zip(host_df[host_col], host_df[hii_col]))
    virus_psi = dict(zip(virus_df[virus_col], virus_df[psi_col]))

    return host_hii, virus_psi


# ===============================
# 2. 数据读取与预处理
# ===============================
def load_and_preprocess_data(filepath):
    df = pd.read_excel(filepath)
    df = df[df['virus_new'].notna()].reset_index(drop=True)
    df = df[['zone', 'Host_species', 'virus_new', 'County']]
    df = df.rename(columns={'Host_species': 'host', 'virus_new': 'virus'})

    df = (
        df.assign(virus=df['virus'].str.split(','))
          .explode('virus')
          .assign(virus=lambda x: x['virus'].str.strip())
          .reset_index(drop=True)
    )

    df[['host', 'virus']] = df[['host', 'virus']].apply(lambda x: x.str.strip())

    return df


# ===============================
# 3. 构建三层加权网络
# ===============================
def build_weighted_three_layer_network(
    df,
    zone_col='zone',
    host_col='host',
    virus_col='virus'
):
    G = nx.Graph()

    # 添加节点
    for col, layer in zip(
        [zone_col, host_col, virus_col],
        ['zone', 'host', 'virus']
    ):
        for n in df[col].unique():
            G.add_node(n, layer=layer)

    # 边权
    zh_weight = df.groupby([zone_col, host_col]).size()
    hv_weight = df.groupby([host_col, virus_col]).size()

    for (z, h), w in zh_weight.items():
        G.add_edge(z, h, weight=w, layer='zone-host')

    for (h, v), w in hv_weight.items():
        G.add_edge(h, v, weight=w, layer='host-virus')

    return G


# ===============================
# 4. 节点属性赋值（host / virus）
# ===============================
def assign_node_attributes(G, host_hii, virus_psi):
    for n, d in G.nodes(data=True):
        if d.get("layer") == "host":
            d["HII"] = host_hii.get(n, np.nan)
        elif d.get("layer") == "virus":
            d["PSI"] = virus_psi.get(n, np.nan)


# ===============================
# 4.5 计算 zone 的连接强度（strength）
# ===============================
def assign_zone_strength(G):
    for n, d in G.nodes(data=True):
        if d.get("layer") == "zone":
            d["strength"] = sum(
                edata.get("weight", 1)
                for _, _, edata in G.edges(n, data=True)
            )


# ===============================
# 5. 节点移除并输出结果
# ===============================
def node_removal_record_each_node(
    G,
    target_layer,
    attribute,
    descending=True,
    out_dir="output/4_remove_node"
):
    # 确保输出目录存在
    os.makedirs(out_dir, exist_ok=True)

    H = copy.deepcopy(G)
    N0 = H.number_of_nodes()

    # 目标节点
    target_nodes = [
        n for n, d in H.nodes(data=True)
        if d.get("layer") == target_layer
        and attribute in d
        and not pd.isna(d[attribute])
    ]

    # 排序
    # ===============================
    # 🔥 关键修改：zone 固定移除顺序
    # ===============================
    if target_layer == 'zone':
        fixed_order = ['SC', 'QTP', 'NNE', 'NW']
        target_nodes = [z for z in fixed_order if z in target_nodes]

    else:
        # host / virus：按属性排序（原逻辑）
        target_nodes = sorted(
            target_nodes,
            key=lambda n: H.nodes[n][attribute],
            reverse=descending
        )

    removed_nodes = []
    removed_frac = []
    giant_component_frac = []

    for i, node in enumerate(target_nodes, start=1):
        H.remove_node(node)
        removed_nodes.append(node)

        if H.number_of_nodes() > 0:
            gcc = max(nx.connected_components(H), key=len)
            Smax = len(gcc)
        else:
            Smax = 0

        removed_frac.append(i / len(target_nodes))
        giant_component_frac.append(Smax / N0)

    df_result = pd.DataFrame({
        "removed_node": removed_nodes,
        "removed_fraction": removed_frac,
        "giant_component_fraction": giant_component_frac
    })

    out_file = os.path.join(
        out_dir,
        f"remove_{target_layer}_by_{attribute}.xlsx"
    )
    df_result.to_excel(out_file, index=False)

    print(f"[OK] {target_layer} removal saved: {out_file}")

    return df_result


# ===============================
# 6. 主流程
# ===============================
def run_attribute_based_robustness_analysis(
    data_file,
    host_index_file,
    virus_index_file
):
    df = load_and_preprocess_data(data_file)

    G = build_weighted_three_layer_network(df)

    host_hii, virus_psi = load_host_virus_indices(
        host_file=host_index_file,
        virus_file=virus_index_file
    )

    assign_node_attributes(G, host_hii, virus_psi)
    assign_zone_strength(G)

    zone_res  = node_removal_record_each_node(G, 'zone',  'strength')
    host_res  = node_removal_record_each_node(G, 'host',  'HII')
    virus_res = node_removal_record_each_node(G, 'virus', 'PSI')

    return zone_res, host_res, virus_res


# ===============================
# 7. 运行
# ===============================
zone_res, host_res, virus_res = run_attribute_based_robustness_analysis(
    data_file=r'input/host_virus-NEW-0313.xlsx',
    host_index_file='output/2_multilayer_centrality/host_centrality_weighted.xlsx',
    virus_index_file='output/2_multilayer_centrality/virus_centrality_weighted.xlsx'
)

[OK] zone removal saved: output/4_remove_node\remove_zone_by_strength.xlsx
[OK] host removal saved: output/4_remove_node\remove_host_by_HII.xlsx
[OK] virus removal saved: output/4_remove_node\remove_virus_by_PSI.xlsx
